# Channel audit: M3 backend check on a Colab or Kaggle GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arshia-HZ/mas-channel-audit/blob/main/notebooks/runner.ipynb)

This notebook runs the whole pipeline on a **small check set** (50 MATH *train*
problems, levels 3 to 5, one replicate) with a real model, so you can:

1. confirm the model, chat template and decoding settings behave,
2. read **30 outputs across every arm** (`review_sample.md`) before the prompts are frozen,
3. see throughput (completion tokens per second) and whether gate G3 holds
   (no context overflow, truncation under 5%),
4. check that a rerun reproduces the same generations.

**Setup.** Colab: *Runtime, Change runtime type, T4 GPU*. Kaggle: *Accelerator GPU T4 x1* and
*Internet on*. Then *Run all*. Every step is **resumable**: finished generations are
skipped, so after a disconnect just run the cells again. If the runtime asks to restart after
the install cell, restart and run again from the top.

**Quick token-limit test.** Set `MAX_TOKENS = 2048` and `ROUND0_ONLY = True` in the settings: only
round 0 runs (150 generations), into its own `_mt2048` directory, and a short report shows the
truncation rate against gate G3.

Nothing here freezes the prompts or touches the study data; those are separate decisions.

## 0. Settings

In [ ]:
MODEL_CONFIG = (
    "configs/models/qwen3-1.7b.yaml"  # or configs/models/qwen2.5-math-1.5b.yaml
)
BACKEND = "vllm"  # "hf" only if vLLM cannot start on this GPU (slower; never mix within a model)
N_PROBLEMS = 50
REPLICATES = 1
REF = "main"  # branch, tag or commit of the repository to run
REPO_URL = "https://github.com/Arshia-HZ/mas-channel-audit.git"
USE_DRIVE = (
    True  # Colab only: keep results in Google Drive so they survive a disconnect
)
SHOW_PROMPT = True  # print the rendered chat prompt (downloads the tokenizer only)
MAX_TOKENS = None  # e.g. 2048 to override the config's max_tokens (own run directory)
ROUND0_ONLY = False  # True: only round 0 plus a truncation report

## 1. Environment

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if IN_COLAB:
    REPO = "/content/mas-channel-audit"
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        RUN_ROOT = "/content/drive/MyDrive/mas-channel-audit"
    else:
        RUN_ROOT = "/content/mas-channel-audit-runs"
elif IN_KAGGLE:
    REPO = "/kaggle/working/mas-channel-audit"
    RUN_ROOT = "/kaggle/working/runs"
else:  # a local checkout
    REPO = os.getcwd()
    if os.path.basename(REPO) == "notebooks":
        REPO = os.path.dirname(REPO)
    RUN_ROOT = os.environ.get("MCA_RUN_ROOT", os.path.join(REPO, "runs"))
os.makedirs(RUN_ROOT, exist_ok=True)


def sh(*cmd, cwd=None, check=True):
    """Run a command, show its output in the cell, raise if it fails."""
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(proc.stdout, end="")
    print(proc.stderr, end="", file=sys.stderr)
    if check and proc.returncode:
        raise RuntimeError(f"{' '.join(cmd)} exited with {proc.returncode}")
    return proc


print("colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local", "| repo:", REPO)
print("results go to:", RUN_ROOT)

In [ ]:
if IN_COLAB or IN_KAGGLE:
    if not os.path.isdir(f"{REPO}/.git"):
        sh("git", "clone", "-q", REPO_URL, REPO)
    sh("git", "fetch", "-q", "--all", cwd=REPO)
    sh("git", "checkout", "-q", REF, cwd=REPO)
    if subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO).returncode == 0:
        sh("git", "pull", "-q", "--ff-only", cwd=REPO)  # on a branch: take the latest
    extra = "vllm" if BACKEND == "vllm" else "hf"
    sh(sys.executable, "-m", "pip", "install", "-q", "-e", f".[{extra},data]", cwd=REPO)

os.chdir(REPO)
sys.path.insert(
    0, f"{REPO}/src"
)  # a running kernel does not see a fresh editable install
sh("git", "log", "--oneline", "-1")

**Optional: Hugging Face token.** Only the gated Llama model needs one (store it as a Colab
secret or Kaggle secret named `HF_TOKEN`). The token is never printed.

In [ ]:
token = os.environ.get("HF_TOKEN")
if not token and IN_COLAB:
    from google.colab import userdata

    try:
        token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        token = None
if not token and IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient

    try:
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:  # the secret is simply not set
        token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF token:", "found" if token else "not set (fine for the Qwen models)")

In [ ]:
import shutil
from importlib import metadata

if shutil.which("nvidia-smi"):
    sh("nvidia-smi", check=False)
for pkg in ("vllm", "torch", "transformers", "numpy", "math-verify", "pydantic"):
    try:
        print(f"{pkg:<14}{metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:<14}not installed")

## 2. Model and prompt

In [ ]:
from channel_audit.models import load_model_config

MODEL_PATH = f"{REPO}/{MODEL_CONFIG}"
model = load_model_config(MODEL_PATH)
if model.backend != BACKEND:
    model = model.model_copy(update={"backend": BACKEND})
slug = model.model_id.split("/")[-1].lower()
RUN = f"{RUN_ROOT}/check_{slug}_{BACKEND}"
if MAX_TOKENS is not None:
    if MAX_TOKENS >= model.max_model_len:
        raise ValueError(
            f"MAX_TOKENS must be below max_model_len={model.max_model_len}"
        )
    model = model.model_copy(update={"max_tokens": MAX_TOKENS})
    RUN += f"_mt{MAX_TOKENS}"
print(model.model_id, "@", model.revision[:12], "|", model.backend, model.dtype)
print(
    "chat template kwargs:",
    model.chat_template_kwargs,
    "| max_tokens:",
    model.max_tokens,
)
print("run directory:", RUN)

In [ ]:
if SHOW_PROMPT:
    sh(sys.executable, "scripts/03_render_prompt.py", "--model-config", MODEL_PATH)

Check the printed prompt: it should be the model's own chat format, and for Qwen3 it must end
with an empty `<think></think>` block (non-thinking mode).

In [ ]:
PROBLEMS = f"{RUN_ROOT}/check_problems.jsonl"
if os.path.exists(PROBLEMS):
    print("using the existing check set:", PROBLEMS)
else:
    sh(
        sys.executable,
        "scripts/03_make_check_problems.py",
        "--n",
        str(N_PROBLEMS),
        "--out",
        PROBLEMS,
    )

## 3. Generate

The model is loaded **once** below (a minute or two) and every step reuses it. Rough cost:
50 problems x 33 generations = about 1,650 generations, several hundred thousand tokens.

In [ ]:
from channel_audit.channels import ARMS
from channel_audit.runner import (
    backend_for_run,
    config_for_model,
    create_run,
    grade_pending,
    run_arm,
    run_failure_messages,
    run_round0,
    run_sc_pool,
    throughput_summary,
)
from channel_audit.schema import Problem
from channel_audit.store import JsonlStore

cfg = config_for_model(model, replicates=REPLICATES, log_prompts=True)
problems = list(JsonlStore(PROBLEMS, Problem, key="problem_id", readonly=True))
ctx = create_run(RUN, cfg, problems, backend_for_run(cfg, problems))  # loads the model
print(f"{len(problems)} problems; {len(ctx.gens)} generations already stored")

In [ ]:
def report(step):
    t = throughput_summary(ctx.out_dir)
    print(
        f"{step}: {len(ctx.gens)} generations stored; "
        f"{t['completion_tokens_per_s']:.0f} completion tokens/s so far"
    )


run_round0(ctx)
report("round 0")
if ROUND0_ONLY:
    from channel_audit.backend_check import format_round0_summary, round0_summary

    grade_pending(ctx)
    print(format_round0_summary(round0_summary(ctx)))
else:
    run_sc_pool(ctx)
    report("self-consistency pool")

In [ ]:
if not ROUND0_ONLY:
    from collections import Counter

    run_failure_messages(ctx)
    report("failure messages")
    print("leakage outcomes:", dict(Counter(r.outcome for r in ctx.leakage)))

In [ ]:
for arm in ARMS:
    if not ROUND0_ONLY:
        run_arm(ctx, arm)
        report(arm)

In [ ]:
print(grade_pending(ctx), "generations graded")
if not ROUND0_ONLY:
    validate_proc = sh(
        sys.executable, "scripts/21_validate_run.py", "--out", RUN, check=False
    )
    print(
        "validate exit code:",
        validate_proc.returncode,
        "(0 ok, 1 errors, 2 gate G3 failed)",
    )

Keep the validate output: it lists the throughput, per-arm truncation and context overflow,
and whether gate G3 holds. For Qwen2.5-Math (4096-token context) watch the overflow column.

## 4. Read the outputs

`review_sample.md` holds 30 outputs spread over every kind of generation, each with the
exact prompt that produced it. Read for: the answer in `\boxed{}`, a `Confidence:` line in
round 0, no copying of peer text, failure messages that name a step but give no answer, and
sensible stopping. Write down anything odd; do **not** edit the prompts yet.

In [ ]:
REVIEW = f"{RUN}/review_sample.md"
if not ROUND0_ONLY:
    if os.path.exists(REVIEW):
        print("already written:", REVIEW)
    else:
        sh(sys.executable, "scripts/03_review_sample.py", "--out", RUN, "--n", "30")
    print(open(REVIEW, encoding="utf-8").read()[:4000])
    if IN_COLAB:
        from google.colab import files

        files.download(REVIEW)

## 5. Reproducibility

Same backend and version, same plan. The first rerun regenerates **round 0 only** with the same
batch composition; the second uses only 10 problems, so batches differ. Text can differ in the
second case because GPU kernels are not batch-invariant, and that difference is what we measure
and document. "Only in A" lines are expected: the reruns do not regenerate the later arms.

In [ ]:
from channel_audit.backend_check import compare_runs


def rerun(subset, name):
    again = create_run(f"{RUN}_repro_{name}", cfg, subset, ctx.backend)
    run_round0(again)
    return again


lines = []
for name, subset in (("same_batch", problems), ("small_batch", problems[:10])):
    if ROUND0_ONLY:
        break
    lines.append(f"== rerun {name}: {len(subset)} problems")
    lines.append(compare_runs(ctx, rerun(subset, name)).format())
if lines:
    repro_text = "\n".join(lines)
    print(repro_text)
    with open(f"{RUN}/repro_report.txt", "w", encoding="utf-8") as fh:
        fh.write(repro_text + "\n")

## 6. Save the results

In [ ]:
archive = shutil.make_archive(f"{RUN}_results", "zip", root_dir=RUN)
print("saved:", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(archive)

**Send back:** the `validate` output (throughput, truncation, overflow, G3), your notes from
reading the 30 outputs, and the reproducibility report. The zip also holds every generation,
prompt and graded record. After you approve the prompts, they get frozen with
`scripts/03_freeze_prompts.py`.